<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/FINANCE_TOPO_GEMMA_INKLING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://openrouter.ai/settings/credits

https://huggingface.co/frankmorales2020/gemma-4-e4b-stl10-topo-2026

In [ ]:
!pip install -q transformers torch openai python-dotenv pyyaml requests numpy pydantic fastapi uvicorn
!pip install -U bitsandbytes>=0.46.1 -q
!pip install unsloth -q

In [ ]:
import requests
import json

def list_openrouter_models():
    """
    Connects to the OpenRouter API to fetch the list of available models
    and prints key details for each model.
    """

    # OpenRouter API endpoint for listing models
    API_URL = "https://openrouter.ai/api/v1/models"

    print(f"Fetching models from: {API_URL}\n")

    try:
        # Send a GET request to the models endpoint
        response = requests.get(API_URL)

        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()

        # Parse the JSON response
        data = response.json()

        # The list of models is contained in the 'data' field
        models = data.get('data', [])

        if not models:
            print("No models found in the API response.")
            return

        # --- Print the results in a formatted way ---
        print(f"Found {len(models)} total models. Key details:\n")
        print("{:<45} {:<30} {:<15}".format("Model ID", "Name", "Context Length"))
        print("-" * 90)

        # Iterate over the model list and print details
        for model in models:
            model_id = model.get('id', 'N/A')
            name = model.get('name', 'N/A')
            context_length = model.get('context_length', 'N/A')

            # Truncate model ID and name for clean printing
            display_id = model_id[:42] + '...' if len(model_id) > 45 else model_id
            display_name = name[:27] + '...' if len(name) > 30 else name

            print("{:<45} {:<30} {:<15}".format(display_id, display_name, context_length))

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while connecting to the OpenRouter API: {e}")
    except json.JSONDecodeError:
        print("Error: Failed to decode JSON response from the API.")

if __name__ == "__main__":
    list_openrouter_models()

## 💰 Ferrari AI - Financial Analysis Agent -CASE1

In [2]:
# ============================================================
# FERRARI AI - FINANCIAL ANALYSIS AGENT
# Complete Agentic Solution for Financial Analysis & Trading
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random
import re

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# CORRECT INKLING MODEL ID
# ============================================================
INKLING_MODEL_ID = "thinkingmachines/inkling"
INKLING_MAX_TOKENS = 2048
INKLING_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ============================================================
# PART 2: INKLING CLIENT
# ============================================================

@dataclass
class InklingResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class InklingClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = INKLING_MODEL_ID,
                 max_tokens: int = INKLING_MAX_TOKENS,
                 temperature: float = INKLING_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Inkling will not work.")
        else:
            print(f"✅ Inkling client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> InklingResponse:
        if not self.api_key:
            return InklingResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return InklingResponse(content=error_content)

            data = response.json()
            message = data['choices'][0]['message']

            content = message.get('content')
            reasoning = message.get('reasoning')

            if not content and reasoning:
                content = reasoning
                reasoning = None

            if not content:
                content = "No response content received."

            return InklingResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return InklingResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: FINANCIAL ANALYSIS AGENT
# ============================================================

class FinanceTaskType(Enum):
    STOCK_PRICE = "stock_price"
    PORTFOLIO_CHECK = "portfolio_check"
    RISK_ASSESSMENT = "risk_assessment"
    TRADE_RECOMMENDATION = "trade_recommendation"
    MARKET_SENTIMENT = "market_sentiment"
    FINANCIAL_RATIO = "financial_ratio"
    INVESTMENT_ADVICE = "investment_advice"
    GENERAL = "general"


@dataclass
class Stock:
    symbol: str
    company_name: str
    sector: str
    current_price: float
    previous_close: float
    day_change: float
    volume: int
    market_cap: float
    pe_ratio: float
    dividend_yield: float
    fifty_two_week_high: float
    fifty_two_week_low: float
    last_updated: datetime

    def to_dict(self) -> Dict:
        return {
            'symbol': self.symbol,
            'company_name': self.company_name,
            'sector': self.sector,
            'current_price': self.current_price,
            'previous_close': self.previous_close,
            'day_change': self.day_change,
            'day_change_percent': (self.day_change / self.previous_close) * 100,
            'volume': self.volume,
            'market_cap': self.market_cap,
            'pe_ratio': self.pe_ratio,
            'dividend_yield': self.dividend_yield,
            'fifty_two_week_high': self.fifty_two_week_high,
            'fifty_two_week_low': self.fifty_two_week_low,
            'last_updated': self.last_updated.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class Portfolio:
    name: str
    stocks: Dict[str, int]  # symbol -> shares
    cash: float
    last_updated: datetime

    def to_dict(self) -> Dict:
        return {
            'name': self.name,
            'stocks': self.stocks,
            'cash': self.cash,
            'last_updated': self.last_updated.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class MarketNews:
    headline: str
    source: str
    category: str
    sentiment: str  # POSITIVE, NEGATIVE, NEUTRAL
    timestamp: datetime
    summary: str


@dataclass
class FinancialResponse:
    task_type: FinanceTaskType
    action_taken: str
    data: Dict
    reasoning: str
    recommendations: List[str] = field(default_factory=list)
    confidence: float = 0.0
    risk_level: str = "MEDIUM"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class FinancialAnalysisAgent:
    """
    Agentic AI System for Financial Analysis and Trading
    Combines Gemma-4 classification with Inkling reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("💰  FINANCIAL ANALYSIS AGENT")
        print("="*60)

        # Initialize Ferrari AI components
        self.gemma = GemmaTOPOCertified()
        self.inkling = InklingClient()

        # Financial database (simulated)
        self.stocks: Dict[str, Stock] = {}
        self.portfolios: Dict[str, Portfolio] = {}
        self.market_news: List[MarketNews] = []
        self._initialize_market_data()
        self._initialize_portfolios()
        self._initialize_news()

        print("="*60)
        print("✅ Financial Analysis Agent Ready!")
        print(f"   📈 Stocks: {len(self.stocks)}")
        print(f"   💼 Portfolios: {len(self.portfolios)}")
        print(f"   📰 News: {len(self.market_news)}")
        print("="*60 + "\n")

    def _initialize_market_data(self):
        """Initialize simulated stock market data."""
        now = datetime.now()

        stocks_data = [
            ("AAPL", "Apple Inc.", "Technology", 175.34, 173.50, 1.84, 85000000, 2.8e12, 28.5, 0.5, 198.23, 165.00, now),
            ("GOOGL", "Alphabet Inc.", "Technology", 141.80, 140.20, 1.60, 32000000, 1.8e12, 24.2, 0.0, 155.00, 120.00, now),
            ("MSFT", "Microsoft Corp.", "Technology", 378.90, 376.50, 2.40, 28000000, 2.9e12, 32.8, 0.8, 400.00, 320.00, now),
            ("AMZN", "Amazon.com Inc.", "Consumer", 185.20, 183.00, 2.20, 35000000, 1.9e12, 45.6, 0.0, 200.00, 160.00, now),
            ("TSLA", "Tesla Inc.", "Automotive", 245.60, 240.00, 5.60, 120000000, 780.0e9, 72.5, 0.0, 280.00, 200.00, now),
            ("JPM", "JPMorgan Chase", "Financial", 155.30, 153.80, 1.50, 15000000, 450.0e9, 12.8, 2.8, 165.00, 135.00, now),
            ("VTI", "Vanguard Total Stock", "ETF", 258.40, 256.50, 1.90, 8000000, 1.2e12, 21.5, 1.5, 270.00, 240.00, now),
            ("BND", "Vanguard Total Bond", "ETF", 72.80, 72.50, 0.30, 5000000, 320.0e9, 15.2, 3.2, 75.00, 70.00, now),
        ]

        for stock_data in stocks_data:
            stock = Stock(
                symbol=stock_data[0],
                company_name=stock_data[1],
                sector=stock_data[2],
                current_price=stock_data[3],
                previous_close=stock_data[4],
                day_change=stock_data[5],
                volume=stock_data[6],
                market_cap=stock_data[7],
                pe_ratio=stock_data[8],
                dividend_yield=stock_data[9],
                fifty_two_week_high=stock_data[10],
                fifty_two_week_low=stock_data[11],
                last_updated=stock_data[12]
            )
            self.stocks[stock.symbol] = stock

    def _initialize_portfolios(self):
        """Initialize simulated investment portfolios."""
        now = datetime.now()

        portfolios_data = [
            (
                "Tech Growth Portfolio",
                {"AAPL": 100, "GOOGL": 50, "MSFT": 75, "AMZN": 30, "TSLA": 20},
                10000.0,
                now
            ),
            (
                "Retirement 2050",
                {"VTI": 200, "BND": 150, "JPM": 50, "AAPL": 30},
                50000.0,
                now
            ),
            (
                "Aggressive Trading",
                {"TSLA": 150, "AMZN": 80, "AAPL": 120, "GOOGL": 40},
                5000.0,
                now
            ),
        ]

        for portfolio_data in portfolios_data:
            portfolio = Portfolio(
                name=portfolio_data[0],
                stocks=portfolio_data[1],
                cash=portfolio_data[2],
                last_updated=portfolio_data[3]
            )
            self.portfolios[portfolio.name] = portfolio

    def _initialize_news(self):
        """Initialize simulated market news."""
        now = datetime.now()

        news_data = [
            MarketNews(
                headline="Fed Signals Rate Cut as Inflation Eases",
                source="Bloomberg",
                category="Macro",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=2),
                summary="Federal Reserve officials signal potential rate cuts as inflation shows signs of cooling."
            ),
            MarketNews(
                headline="Tech Sector Rally Continues on AI Optimism",
                source="Reuters",
                category="Sector",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=4),
                summary="Technology stocks extend gains as AI investments drive investor enthusiasm."
            ),
            MarketNews(
                headline="Oil Prices Drop on Supply Concerns",
                source="CNBC",
                category="Commodities",
                sentiment="NEGATIVE",
                timestamp=now - timedelta(hours=6),
                summary="Crude oil prices fall as supply concerns ease, benefiting transportation and consumer sectors."
            ),
            MarketNews(
                headline="Tesla Beats Delivery Estimates for Q2",
                source="Business Insider",
                category="Earnings",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=8),
                summary="Tesla reports stronger-than-expected vehicle deliveries, boosting investor confidence."
            ),
            MarketNews(
                headline="Market Uncertainty Ahead of Earnings Season",
                source="Financial Times",
                category="Market",
                sentiment="NEUTRAL",
                timestamp=now - timedelta(hours=10),
                summary="Investors cautious as earnings season approaches, with mixed signals from economic data."
            ),
        ]

        self.market_news = news_data

    def identify_task(self, query: str) -> Dict:
        """Classify the financial query type."""
        query_lower = query.lower()

        task_mapping = {
            'price': FinanceTaskType.STOCK_PRICE,
            'stock': FinanceTaskType.STOCK_PRICE,
            'quote': FinanceTaskType.STOCK_PRICE,
            'portfolio': FinanceTaskType.PORTFOLIO_CHECK,
            'holdings': FinanceTaskType.PORTFOLIO_CHECK,
            'risk': FinanceTaskType.RISK_ASSESSMENT,
            'volatility': FinanceTaskType.RISK_ASSESSMENT,
            'trade': FinanceTaskType.TRADE_RECOMMENDATION,
            'buy': FinanceTaskType.TRADE_RECOMMENDATION,
            'sell': FinanceTaskType.TRADE_RECOMMENDATION,
            'sentiment': FinanceTaskType.MARKET_SENTIMENT,
            'news': FinanceTaskType.MARKET_SENTIMENT,
            'pe': FinanceTaskType.FINANCIAL_RATIO,
            'ratio': FinanceTaskType.FINANCIAL_RATIO,
            'dividend': FinanceTaskType.FINANCIAL_RATIO,
            'invest': FinanceTaskType.INVESTMENT_ADVICE,
            'advice': FinanceTaskType.INVESTMENT_ADVICE,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': FinanceTaskType.GENERAL, 'confidence': 0.6}

    def _get_stock(self, symbol: str) -> Optional[Dict]:
        """Get stock data by symbol."""
        symbol_upper = symbol.upper()
        if symbol_upper in self.stocks:
            return self.stocks[symbol_upper].to_dict()
        return None

    def _get_portfolio(self, name: str) -> Optional[Dict]:
        """Get portfolio by name."""
        for portfolio_name, portfolio in self.portfolios.items():
            if name.lower() in portfolio_name.lower():
                return portfolio.to_dict()
        return None

    def _get_stocks_by_sector(self, sector: str) -> List[Dict]:
        """Get stocks by sector."""
        results = []
        for stock in self.stocks.values():
            if sector.lower() in stock.sector.lower():
                results.append(stock.to_dict())
        return results

    def _calculate_portfolio_value(self, portfolio: Portfolio) -> float:
        """Calculate total portfolio value."""
        total = portfolio.cash
        for symbol, shares in portfolio.stocks.items():
            if symbol in self.stocks:
                total += shares * self.stocks[symbol].current_price
        return total

    def _get_portfolio_performance(self, portfolio: Portfolio) -> Dict:
        """Calculate portfolio performance metrics."""
        total_value = self._calculate_portfolio_value(portfolio)
        stock_values = {}
        for symbol, shares in portfolio.stocks.items():
            if symbol in self.stocks:
                stock_values[symbol] = shares * self.stocks[symbol].current_price

        return {
            'total_value': total_value,
            'cash': portfolio.cash,
            'stock_values': stock_values,
            'stock_allocations': {s: v/total_value for s, v in stock_values.items()},
            'num_positions': len(portfolio.stocks)
        }

    def _get_market_sentiment(self) -> Dict:
        """Analyze market sentiment from news."""
        sentiments = {'POSITIVE': 0, 'NEGATIVE': 0, 'NEUTRAL': 0}
        for news in self.market_news:
            sentiments[news.sentiment] += 1

        total = len(self.market_news)
        if total == 0:
            return {'overall': 'NEUTRAL', 'positive_pct': 0, 'negative_pct': 0}

        return {
            'overall': 'POSITIVE' if sentiments['POSITIVE'] > sentiments['NEGATIVE'] else 'NEGATIVE' if sentiments['NEGATIVE'] > sentiments['POSITIVE'] else 'NEUTRAL',
            'positive_pct': sentiments['POSITIVE'] / total * 100,
            'negative_pct': sentiments['NEGATIVE'] / total * 100,
            'neutral_pct': sentiments['NEUTRAL'] / total * 100,
            'total_news': total
        }

    def process_query(self, query: str) -> FinancialResponse:
        """
        Main processing method for the Financial Analysis Agent.
        """
        print(f"\n💰 Financial Query: {query}")
        print("-" * 50)

        # Step 1: Identify task type
        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # Step 2: Extract symbols and names
        symbols = re.findall(r'[A-Z]{1,5}', query.upper())
        symbols = [s for s in symbols if s in self.stocks]

        portfolio_match = re.search(r'(Tech Growth Portfolio|Retirement 2050|Aggressive Trading)', query)
        portfolio_name = portfolio_match.group(0) if portfolio_match else None

        # Step 3: Retrieve data based on task type
        data = {}
        action_taken = ""
        recommendations = []
        risk_level = "MEDIUM"

        if task_type == FinanceTaskType.STOCK_PRICE:
            if symbols:
                stock_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        stock_data.append(stock)
                data['stocks'] = stock_data
                action_taken = f"Retrieved price data for {', '.join(symbols)}"
            else:
                data['all_stocks'] = [stock.to_dict() for stock in self.stocks.values()]
                action_taken = "Retrieved all stock prices"

        elif task_type == FinanceTaskType.PORTFOLIO_CHECK:
            if portfolio_name:
                portfolio = self._get_portfolio(portfolio_name)
                if portfolio:
                    data['portfolio'] = portfolio
                    # Get performance metrics
                    for p in self.portfolios.values():
                        if portfolio_name.lower() in p.name.lower():
                            perf = self._get_portfolio_performance(p)
                            data['performance'] = perf
                            break
                    action_taken = f"Retrieved portfolio: {portfolio_name}"
                else:
                    data['all_portfolios'] = [p.to_dict() for p in self.portfolios.values()]
                    data['performance'] = {
                        p.name: self._get_portfolio_performance(p)
                        for p in self.portfolios.values()
                    }
                    action_taken = "Retrieved all portfolios"
            else:
                data['all_portfolios'] = [p.to_dict() for p in self.portfolios.values()]
                data['performance'] = {
                    p.name: self._get_portfolio_performance(p)
                    for p in self.portfolios.values()
                }
                action_taken = "Retrieved all portfolios"

        elif task_type == FinanceTaskType.RISK_ASSESSMENT:
            if symbols:
                risk_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        # Calculate risk metrics
                        price_range = stock['fifty_two_week_high'] - stock['fifty_two_week_low']
                        volatility = price_range / stock['fifty_two_week_low'] * 100
                        risk_data.append({
                            'symbol': symbol,
                            'volatility': f"{volatility:.1f}%",
                            'current_position': f"{stock['current_price']}/{stock['fifty_two_week_high']} ({stock['current_price']/stock['fifty_two_week_high']*100:.1f}% of high)",
                            'risk_level': 'HIGH' if volatility > 30 else 'MEDIUM' if volatility > 20 else 'LOW'
                        })
                data['risk_analysis'] = risk_data
                action_taken = f"Assessed risk for {', '.join(symbols)}"
            elif portfolio_name:
                for p in self.portfolios.values():
                    if portfolio_name.lower() in p.name.lower():
                        perf = self._get_portfolio_performance(p)
                        # Calculate portfolio risk (simplified)
                        avg_volatility = 0
                        for symbol in p.stocks:
                            if symbol in self.stocks:
                                stock = self.stocks[symbol]
                                price_range = stock.fifty_two_week_high - stock.fifty_two_week_low
                                avg_volatility += (price_range / stock.fifty_two_week_low * 100) * (p.stocks[symbol] / sum(p.stocks.values()))
                        data['portfolio'] = p.to_dict()
                        data['portfolio_risk'] = {
                            'volatility': f"{avg_volatility:.1f}%",
                            'risk_level': 'HIGH' if avg_volatility > 30 else 'MEDIUM' if avg_volatility > 20 else 'LOW',
                            'num_positions': len(p.stocks)
                        }
                        action_taken = f"Assessed risk for portfolio: {portfolio_name}"
                        break
            else:
                data['risk_summary'] = "Please specify a stock symbol or portfolio name"
                action_taken = "Risk assessment requires symbol or portfolio"

        elif task_type == FinanceTaskType.TRADE_RECOMMENDATION:
            if symbols:
                trade_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        # Simple trade analysis
                        price_position = (stock['current_price'] - stock['fifty_two_week_low']) / (stock['fifty_two_week_high'] - stock['fifty_two_week_low']) * 100
                        if price_position < 30:
                            recommendation = "BUY"
                            reason = "Trading near 52-week low"
                        elif price_position > 70:
                            recommendation = "SELL"
                            reason = "Trading near 52-week high"
                        else:
                            recommendation = "HOLD"
                            reason = "Mid-range valuation"
                        trade_data.append({
                            'symbol': symbol,
                            'recommendation': recommendation,
                            'reason': reason,
                            'price_position': f"{price_position:.0f}% of 52-week range"
                        })
                data['trade_recommendations'] = trade_data
                action_taken = f"Generated trade recommendations for {', '.join(symbols)}"
            else:
                data['trade_advice'] = "Please specify a stock symbol for trade recommendations"
                action_taken = "Trade recommendation requires symbol"

        elif task_type == FinanceTaskType.MARKET_SENTIMENT:
            sentiment = self._get_market_sentiment()
            data['market_sentiment'] = sentiment
            data['recent_news'] = [{
                'headline': n.headline,
                'source': n.source,
                'sentiment': n.sentiment,
                'summary': n.summary
            } for n in self.market_news[:3]]
            action_taken = "Retrieved market sentiment and news"

        elif task_type == FinanceTaskType.FINANCIAL_RATIO:
            if symbols:
                ratio_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        ratio_data.append({
                            'symbol': symbol,
                            'company': stock['company_name'],
                            'P/E Ratio': stock['pe_ratio'],
                            'Dividend Yield': f"{stock['dividend_yield']:.2f}%",
                            'Market Cap': f"${stock['market_cap']:.1e}",
                            'Volume': f"{stock['volume']:,}",
                            'PE vs Sector': 'Above average' if stock['pe_ratio'] > 25 else 'Below average' if stock['pe_ratio'] < 15 else 'Average'
                        })
                data['financial_ratios'] = ratio_data
                action_taken = f"Retrieved financial ratios for {', '.join(symbols)}"
            else:
                data['ratios'] = "Please specify a stock symbol for financial ratios"
                action_taken = "Financial ratio check requires symbol"

        elif task_type == FinanceTaskType.INVESTMENT_ADVICE:
            if portfolio_name:
                for p in self.portfolios.values():
                    if portfolio_name.lower() in p.name.lower():
                        perf = self._get_portfolio_performance(p)
                        data['portfolio'] = p.to_dict()
                        data['performance'] = perf
                        data['advice_context'] = f"Portfolio: {portfolio_name}, Value: ${perf['total_value']:.2f}"
                        break
            else:
                data['advice_context'] = "General investment advice requested"
                data['market_context'] = self._get_market_sentiment()
            action_taken = "Generated investment advice"

        else:  # GENERAL
            data['message'] = "General financial inquiry received"
            action_taken = "General query processed"

        # Step 4: Generate reasoning using Inkling
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        inkling_response = self.inkling.query(reasoning_prompt)

        print(f"🧠 Inkling Reasoning: {inkling_response.content[:200]}...")

        # Step 5: Build response
        return FinancialResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=inkling_response.content,
            recommendations=self._extract_recommendations(inkling_response.content),
            confidence=confidence,
            risk_level=risk_level,
            timestamp=datetime.now().isoformat()
        )

    def _generate_reasoning_prompt(self, query: str, task_type: FinanceTaskType, data: Dict) -> str:
        """Generate a prompt for Inkling to reason about the financial situation."""
        return f"""
        You are a Financial Analysis AI Assistant providing investment and trading advice.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Financial Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clear financial analysis of the situation
        2. Key insights from the data
        3. Recommended actions or decisions
        4. Risk assessment and considerations
        5. A concise summary for the investor/trader

        Your response should be professional, data-driven, and actionable.
        Always include a disclaimer that this is AI-assisted analysis and not personalized financial advice.
        """

    def _extract_recommendations(self, reasoning: str) -> List[str]:
        """Extract recommendations from the reasoning text."""
        recommendations = []
        lines = reasoning.split('\n')
        for line in lines:
            if any(keyword in line.lower() for keyword in ['recommend', 'advise', 'suggest', 'should', 'consider']):
                cleaned = line.strip().lstrip('•*-+0123456789. ')
                if len(cleaned) > 10 and cleaned not in recommendations:
                    recommendations.append(cleaned[:200])
        return recommendations[:5]

    def handle_query(self, query: str) -> str:
        """
        Simplified method to handle a query and return a readable response.
        """
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append("💰 Financial Analysis Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        output.append(f"⚠️ Risk Level: {response.risk_level}")
        output.append("-" * 60)
        output.append("📊 Analysis:")
        output.append(response.reasoning)
        if response.recommendations:
            output.append("-" * 60)
            output.append("💡 Recommendations:")
            for i, rec in enumerate(response.recommendations, 1):
                output.append(f"   {i}. {rec}")
        output.append("=" * 60)
        output.append("⚠️ DISCLAIMER: This is AI-assisted financial analysis.")
        output.append("   Always consult with a qualified financial advisor.")
        output.append("   This system does not replace professional financial advice.")
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION
# ============================================================

def run_finance_demo():
    """Demonstrate the Financial Analysis Agent in action."""
    print("\n" + "="*60)
    print("💰  FINANCIAL ANALYSIS AGENT")
    print("   Complete Agentic Solution for Financial Analysis & Trading")
    print("="*60 + "\n")

    agent = FinancialAnalysisAgent()

    test_queries = [
        "What is the current price of AAPL and TSLA?",
        "Show me the Tech Growth Portfolio",
        "Assess risk for MSFT and AMZN",
        "Should I buy or sell GOOGL?",
        "What is the market sentiment today?",
        "What are the financial ratios for JPM?",
    ]

    for i, query in enumerate(test_queries[:5], 1):
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# PART 5: INTERACTIVE FINANCIAL CHAT
# ============================================================

def interactive_finance():
    """Interactive chat with the Financial Analysis Agent."""
    print("\n" + "="*60)
    print("💰  FINANCIAL AGENT - Interactive Investment Analysis")
    print("="*60)
    print("Type your query below. Type 'exit' to quit.")
    print("")
    print("📚 Sample Queries:")
    print("   • What is the price of AAPL?")
    print("   • Show me the Tech Growth Portfolio")
    print("   • Assess risk for MSFT")
    print("   • Should I buy or sell GOOGL?")
    print("   • What is the market sentiment today?")
    print("   • Financial ratios for JPM")
    print("   • Investment advice for Retirement 2050")
    print("-"*60 + "\n")
    print("⚠️ DISCLAIMER: This is AI-assisted financial analysis only.")
    print("   Always consult with a qualified financial advisor.")
    print("   This system does not replace professional financial advice.")
    print("-"*60 + "\n")

    agent = FinancialAnalysisAgent()

    while True:
        try:
            query = input("\n💰 You: ").strip()
            if query.lower() in ['exit', 'quit', 'q']:
                print("\n👋 Financial Agent signing off!")
                break

            if not query:
                continue

            result = agent.handle_query(query)
            print(result)

        except KeyboardInterrupt:
            print("\n\n👋 Financial Agent signing off!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")


# ============================================================
# PART 6: CONFIGURATION
# ============================================================

config = {
    'gemma_model': 'frankmorales2020/gemma-4-e4b-stl10-topo-2026',
    'gemma_base': 'frankmorales2020/gemma-4-e4b-unesco-optimized',
    'inkling_model': INKLING_MODEL_ID,
    'inkling_max_tokens': INKLING_MAX_TOKENS,
    'inkling_temperature': INKLING_TEMPERATURE,
    'api_key_configured': bool(OPENROUTER_API_KEY),
    'system': 'Financial Analysis Agent',
    'version': '1.0.0',
    'stocks': 8,
    'portfolios': 3,
    'news_items': 5
}

print("\n📦 Configuration:")
print(json.dumps(config, indent=2))

# ============================================================
# RUN THE FINANCE DEMO
# ============================================================

# ✅ RUN THE DEMO
run_finance_demo()

# Uncomment to use interactive chat instead:
# interactive_finance()

✅ API key loaded from Colab secrets! Ending: ****f808

📦 Configuration:
{
  "gemma_model": "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
  "gemma_base": "frankmorales2020/gemma-4-e4b-unesco-optimized",
  "inkling_model": "thinkingmachines/inkling",
  "inkling_max_tokens": 2048,
  "inkling_temperature": 0.7,
  "api_key_configured": true,
  "system": "Financial Analysis Agent",
  "version": "1.0.0",
  "stocks": 8,
  "portfolios": 3,
  "news_items": 5
}

💰  FINANCIAL ANALYSIS AGENT
   Complete Agentic Solution for Financial Analysis & Trading


💰  FINANCIAL ANALYSIS AGENT

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...


config.json:   0%|          | 0.00/523 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/6.95k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...


topo_trained_parts_gemma_5runs.pt: reconstructing file:   0%|          |  0.00B / 2.68GB            

topo_trained_parts_gemma_5runs.pt: downloading bytes:           |  0.00B            

   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Inkling client initialized
   Model: thinkingmachines/inkling
   Max Tokens: 2048
   Temperature: 0.7
✅ Financial Analysis Agent Ready!
   📈 Stocks: 8
   💼 Portfolios: 3
   📰 News: 5


TEST 1

💰 Financial Query: What is the current price of AAPL and TSLA?
--------------------------------------------------
📋 Task Identification: stock_price (confidence: 90.00%)
🧠 Inkling Reasoning: **Financial Analysis: AAPL vs. TSLA (As of 2026-08-04 11:44)**

| Metric | **AAPL** (Apple Inc.) | **TSLA** (Tesla Inc.) |
|---|---|---|
| **Current Price** | **$175.34** (+1.06%) | **$245.60** (+2.33...
💰 Financial Analysis Response
📋 Task: stock_price
✅ Action: Retrieved price data for AAPL, TSLA
🔒 Confidence: 90.00%
⚠️ Risk Level: MEDIUM
----------------------------------

## 💰 Ferrari AI - Financial Analysis Agent -CASE2

In [2]:
# ============================================================
# FERRARI AI - FINANCIAL ANALYSIS AGENT
# Complete Agentic Solution for Financial Analysis & Trading
# ============================================================

import os
import json
import torch
import torch.nn as nn
import requests
import contextlib
import io
from typing import Dict, Any, Optional, List
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
import random
import re

# ------------------------------------------------------------------
# API Key from Colab userdata
# ------------------------------------------------------------------
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    if OPENROUTER_API_KEY:
        print(f"✅ API key loaded from Colab secrets! Ending: ****{OPENROUTER_API_KEY[-4:]}")
    else:
        print("⚠️ OPENROUTER_API_KEY not found in Colab secrets.")
        OPENROUTER_API_KEY = ""
except Exception as e:
    print(f"⚠️ Could not load from Colab secrets: {e}")
    OPENROUTER_API_KEY = ""

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# ============================================================
# CORRECT INKLING MODEL ID
# ============================================================
INKLING_MODEL_ID = "thinkingmachines/inkling"
INKLING_MAX_TOKENS = 2048
INKLING_TEMPERATURE = 0.7

# ============================================================
# PART 1: GEMMA-4 E4B TOPO-2026 CLASSIFIER (CF-Free)
# ============================================================

class GemmaTopoClassifier(nn.Module):
    def __init__(self, vision_model, hidden_size=2560):
        super().__init__()
        self.vision_model = vision_model
        self.hidden_size = hidden_size
        self.classifier_A = nn.Linear(hidden_size, 2)
        self.classifier_B = nn.Linear(hidden_size, 2)
        self.classifier_C = nn.Linear(hidden_size, 2)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.vision_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        if hasattr(outputs, 'hidden_states'):
            hidden_states = outputs.hidden_states[-1]
        else:
            hidden_states = outputs.last_hidden_state
        hidden_states = hidden_states.float()
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)
        else:
            pooled = hidden_states.mean(dim=1)
        head = getattr(self, f'classifier_{self.current_task}')
        return head(pooled)

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C')
        self.current_task = task


class GemmaTOPOCertified:
    TASK_LABELS = {
        'A': ['Animal', 'Vehicle'],
        'B': ['Natural', 'Man-Made'],
        'C': ['Living', 'Non-Living']
    }

    TASK_DESCRIPTIONS = {
        'A': 'Animal vs Vehicle',
        'B': 'Natural vs Man-Made',
        'C': 'Living vs Non-Living'
    }

    def __init__(self,
                 repo_id: str = "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
                 base_model: str = "frankmorales2020/gemma-4-e4b-unesco-optimized",
                 device: str = "cuda",
                 max_length: int = 64):

        self.repo_id = repo_id
        self.base_model_name = base_model
        self.device = torch.device(device if torch.cuda.is_available() and device == "cuda" else "cpu")
        self.max_length = max_length
        self.model = None
        self.tokenizer = None
        self._certification_info = {}

        print(f"\n📥 Loading Gemma-4 TOPO-2026 Certified Model...")
        print(f"   Model: {repo_id}")
        print(f"   Device: {self.device}")

        self._load_model()

    def _load_model(self):
        try:
            from transformers import AutoTokenizer
            from huggingface_hub import hf_hub_download

            print("\n📥 Loading tokenizer...")
            self.tokenizer = AutoTokenizer.from_pretrained(self.repo_id, trust_remote_code=True)
            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token
            print(f"   ✅ Tokenizer loaded. Vocab size: {len(self.tokenizer)}")

            print("\n👁️ Loading vision model...")
            vision_model = None

            try:
                with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                    from unsloth import FastVisionModel
                    vision_model, _ = FastVisionModel.from_pretrained(
                        self.base_model_name,
                        load_in_4bit=True,
                        dtype=torch.bfloat16,
                        device_map="auto",
                    )
                    FastVisionModel.for_inference(vision_model)
                print("   ✅ Gemma loaded (Unsloth)")
            except:
                from transformers import AutoModelForCausalLM
                vision_model = AutoModelForCausalLM.from_pretrained(
                    self.base_model_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    trust_remote_code=True
                )
                print("   ✅ Gemma loaded (Transformers)")

            vision_model = vision_model.to(self.device)
            for param in vision_model.parameters():
                param.requires_grad = False

            print("\n📥 Downloading trained weights...")
            ckpt_path = hf_hub_download(self.repo_id, "topo_trained_parts_gemma_5runs.pt")
            ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            print(f"   ✅ Checkpoint loaded! Task C: {ckpt['best_acc_c']*100:.1f}%")

            print("\n🏗️ Building classifier...")
            hidden_size = ckpt['hidden_size']
            self.model = GemmaTopoClassifier(vision_model, hidden_size).to(self.device)

            self.model.classifier_A.load_state_dict(ckpt["classifier_A"])
            self.model.classifier_B.load_state_dict(ckpt["classifier_B"])
            self.model.classifier_C.load_state_dict(ckpt["classifier_C"])

            with torch.no_grad():
                emb_weight = ckpt["embed_tokens_weight"].to(self.device)
                embed_layer = vision_model.get_input_embeddings()
                if emb_weight.shape != embed_layer.weight.shape:
                    if emb_weight.shape[0] < embed_layer.weight.shape[0]:
                        pad_size = embed_layer.weight.shape[0] - emb_weight.shape[0]
                        pad = torch.randn(pad_size, emb_weight.shape[1], device=self.device)
                        emb_weight = torch.cat([emb_weight, pad], dim=0)
                    else:
                        emb_weight = emb_weight[:embed_layer.weight.shape[0]]
                embed_layer.weight.copy_(emb_weight)

            self.model.eval()
            self._certification_info = {
                'standard': 'TOPO-2026',
                'runs': '5/5',
                'task_c_accuracy': f"{ckpt['best_acc_c']*100:.1f}%",
                'forgetting': '0.48%',
                's_narrow': '5.970999999965',
                'status': '✅ CERTIFIED'
            }

            print("   ✅ Model ready!")
            print("\n📊 Certification:")
            for key, value in self._certification_info.items():
                print(f"   {key}: {value}")

        except Exception as e:
            print(f"❌ Failed to load Gemma-4: {e}")
            raise

    def classify(self, text: str, task: str = 'C') -> Dict:
        if self.model is None:
            return self._mock_classify(text, task)

        self.model.switch_task(task)
        tokens = self.tokenizer(
            [text],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        ).to(self.device)

        with torch.no_grad():
            logits = self.model(tokens.input_ids, tokens.attention_mask)
            probs = torch.softmax(logits, dim=1)[0]
            pred_idx = int(torch.argmax(probs))
            confidence = float(probs[pred_idx])

        labels = self.TASK_LABELS[task]
        label = labels[pred_idx]

        return {
            'label': label,
            'confidence': confidence,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: float(probs[0]), labels[1]: float(probs[1])},
            'cf_free': True,
            'memory_guarantee': '0% catastrophic forgetting (TOPO-2026)',
            'certification': self._certification_info
        }

    def _mock_classify(self, text: str, task: str = 'C') -> Dict:
        labels = self.TASK_LABELS[task]
        return {
            'label': labels[0],
            'confidence': 0.95,
            'task': task,
            'task_description': self.TASK_DESCRIPTIONS[task],
            'probabilities': {labels[0]: 0.95, labels[1]: 0.05},
            'cf_free': True,
            'memory_guarantee': 'Mock mode',
            'certification': {'status': '⚠️ MOCK MODE'}
        }


# ============================================================
# PART 2: INKLING CLIENT
# ============================================================

@dataclass
class InklingResponse:
    content: str
    reasoning: Optional[str] = None
    raw_response: Optional[Dict] = None
    model: Optional[str] = None
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class InklingClient:
    def __init__(self,
                 api_key: Optional[str] = None,
                 model: str = INKLING_MODEL_ID,
                 max_tokens: int = INKLING_MAX_TOKENS,
                 temperature: float = INKLING_TEMPERATURE):
        self.api_key = api_key or OPENROUTER_API_KEY
        self.model = model
        self.max_tokens = max_tokens
        self.temperature = temperature
        self.base_url = "https://openrouter.ai/api/v1"

        if not self.api_key:
            print("⚠️ No API key found. Inkling will not work.")
        else:
            print(f"✅ Inkling client initialized")
            print(f"   Model: {model}")
            print(f"   Max Tokens: {max_tokens}")
            print(f"   Temperature: {temperature}")

        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }

    def query(self,
              prompt: str,
              temperature: Optional[float] = None,
              max_tokens: Optional[int] = None) -> InklingResponse:
        if not self.api_key:
            return InklingResponse(
                content="ERROR: No API key. Please add OPENROUTER_API_KEY to Colab secrets."
            )

        temp = temperature if temperature is not None else self.temperature
        max_tok = max_tokens if max_tokens is not None else self.max_tokens

        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temp,
            "max_tokens": max_tok,
            "reasoning": {"enabled": True}
        }

        try:
            response = requests.post(
                f"{self.base_url}/chat/completions",
                headers=self.headers,
                json=payload,
                timeout=60
            )

            if response.status_code != 200:
                error_content = f"API Error ({response.status_code}): {response.text[:200]}"
                return InklingResponse(content=error_content)

            data = response.json()
            message = data['choices'][0]['message']

            content = message.get('content')
            reasoning = message.get('reasoning')

            if not content and reasoning:
                content = reasoning
                reasoning = None

            if not content:
                content = "No response content received."

            return InklingResponse(
                content=content,
                reasoning=reasoning,
                raw_response=data,
                model=data.get('model')
            )

        except Exception as e:
            return InklingResponse(content=f"Error: {str(e)}")


# ============================================================
# PART 3: FINANCIAL ANALYSIS AGENT
# ============================================================

class FinanceTaskType(Enum):
    STOCK_PRICE = "stock_price"
    PORTFOLIO_CHECK = "portfolio_check"
    RISK_ASSESSMENT = "risk_assessment"
    TRADE_RECOMMENDATION = "trade_recommendation"
    MARKET_SENTIMENT = "market_sentiment"
    FINANCIAL_RATIO = "financial_ratio"
    INVESTMENT_ADVICE = "investment_advice"
    GENERAL = "general"


@dataclass
class Stock:
    symbol: str
    company_name: str
    sector: str
    current_price: float
    previous_close: float
    day_change: float
    volume: int
    market_cap: float
    pe_ratio: float
    dividend_yield: float
    fifty_two_week_high: float
    fifty_two_week_low: float
    last_updated: datetime

    def to_dict(self) -> Dict:
        return {
            'symbol': self.symbol,
            'company_name': self.company_name,
            'sector': self.sector,
            'current_price': self.current_price,
            'previous_close': self.previous_close,
            'day_change': self.day_change,
            'day_change_percent': (self.day_change / self.previous_close) * 100,
            'volume': self.volume,
            'market_cap': self.market_cap,
            'pe_ratio': self.pe_ratio,
            'dividend_yield': self.dividend_yield,
            'fifty_two_week_high': self.fifty_two_week_high,
            'fifty_two_week_low': self.fifty_two_week_low,
            'last_updated': self.last_updated.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class Portfolio:
    name: str
    stocks: Dict[str, int]  # symbol -> shares
    cash: float
    last_updated: datetime

    def to_dict(self) -> Dict:
        return {
            'name': self.name,
            'stocks': self.stocks,
            'cash': self.cash,
            'last_updated': self.last_updated.strftime('%Y-%m-%d %H:%M')
        }


@dataclass
class MarketNews:
    headline: str
    source: str
    category: str
    sentiment: str  # POSITIVE, NEGATIVE, NEUTRAL
    timestamp: datetime
    summary: str


@dataclass
class FinancialResponse:
    task_type: FinanceTaskType
    action_taken: str
    data: Dict
    reasoning: str
    recommendations: List[str] = field(default_factory=list)
    confidence: float = 0.0
    risk_level: str = "MEDIUM"
    timestamp: str = field(default_factory=lambda: datetime.now().isoformat())


class FinancialAnalysisAgent:
    """
    Agentic AI System for Financial Analysis and Trading
    Combines Gemma-4 classification with Inkling reasoning
    """

    def __init__(self):
        print("\n" + "="*60)
        print("💰  FINANCIAL ANALYSIS AGENT")
        print("="*60)

        # Initialize Ferrari AI components
        self.gemma = GemmaTOPOCertified()
        self.inkling = InklingClient()

        # Financial database (simulated)
        self.stocks: Dict[str, Stock] = {}
        self.portfolios: Dict[str, Portfolio] = {}
        self.market_news: List[MarketNews] = []
        self._initialize_market_data()
        self._initialize_portfolios()
        self._initialize_news()

        print("="*60)
        print("✅ Financial Analysis Agent Ready!")
        print(f"   📈 Stocks: {len(self.stocks)}")
        print(f"   💼 Portfolios: {len(self.portfolios)}")
        print(f"   📰 News: {len(self.market_news)}")
        print("="*60 + "\n")

    def _initialize_market_data(self):
        """Initialize simulated stock market data."""
        now = datetime.now()

        stocks_data = [
            ("AAPL", "Apple Inc.", "Technology", 175.34, 173.50, 1.84, 85000000, 2.8e12, 28.5, 0.5, 198.23, 165.00, now),
            ("GOOGL", "Alphabet Inc.", "Technology", 141.80, 140.20, 1.60, 32000000, 1.8e12, 24.2, 0.0, 155.00, 120.00, now),
            ("MSFT", "Microsoft Corp.", "Technology", 378.90, 376.50, 2.40, 28000000, 2.9e12, 32.8, 0.8, 400.00, 320.00, now),
            ("AMZN", "Amazon.com Inc.", "Consumer", 185.20, 183.00, 2.20, 35000000, 1.9e12, 45.6, 0.0, 200.00, 160.00, now),
            ("TSLA", "Tesla Inc.", "Automotive", 245.60, 240.00, 5.60, 120000000, 780.0e9, 72.5, 0.0, 280.00, 200.00, now),
            ("JPM", "JPMorgan Chase", "Financial", 155.30, 153.80, 1.50, 15000000, 450.0e9, 12.8, 2.8, 165.00, 135.00, now),
            ("VTI", "Vanguard Total Stock", "ETF", 258.40, 256.50, 1.90, 8000000, 1.2e12, 21.5, 1.5, 270.00, 240.00, now),
            ("BND", "Vanguard Total Bond", "ETF", 72.80, 72.50, 0.30, 5000000, 320.0e9, 15.2, 3.2, 75.00, 70.00, now),
        ]

        for stock_data in stocks_data:
            stock = Stock(
                symbol=stock_data[0],
                company_name=stock_data[1],
                sector=stock_data[2],
                current_price=stock_data[3],
                previous_close=stock_data[4],
                day_change=stock_data[5],
                volume=stock_data[6],
                market_cap=stock_data[7],
                pe_ratio=stock_data[8],
                dividend_yield=stock_data[9],
                fifty_two_week_high=stock_data[10],
                fifty_two_week_low=stock_data[11],
                last_updated=stock_data[12]
            )
            self.stocks[stock.symbol] = stock

    def _initialize_portfolios(self):
        """Initialize simulated investment portfolios."""
        now = datetime.now()

        portfolios_data = [
            (
                "Tech Growth Portfolio",
                {"AAPL": 100, "GOOGL": 50, "MSFT": 75, "AMZN": 30, "TSLA": 20},
                10000.0,
                now
            ),
            (
                "Retirement 2050",
                {"VTI": 200, "BND": 150, "JPM": 50, "AAPL": 30},
                50000.0,
                now
            ),
            (
                "Aggressive Trading",
                {"TSLA": 150, "AMZN": 80, "AAPL": 120, "GOOGL": 40},
                5000.0,
                now
            ),
        ]

        for portfolio_data in portfolios_data:
            portfolio = Portfolio(
                name=portfolio_data[0],
                stocks=portfolio_data[1],
                cash=portfolio_data[2],
                last_updated=portfolio_data[3]
            )
            self.portfolios[portfolio.name] = portfolio

    def _initialize_news(self):
        """Initialize simulated market news."""
        now = datetime.now()

        news_data = [
            MarketNews(
                headline="Fed Signals Rate Cut as Inflation Eases",
                source="Bloomberg",
                category="Macro",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=2),
                summary="Federal Reserve officials signal potential rate cuts as inflation shows signs of cooling."
            ),
            MarketNews(
                headline="Tech Sector Rally Continues on AI Optimism",
                source="Reuters",
                category="Sector",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=4),
                summary="Technology stocks extend gains as AI investments drive investor enthusiasm."
            ),
            MarketNews(
                headline="Oil Prices Drop on Supply Concerns",
                source="CNBC",
                category="Commodities",
                sentiment="NEGATIVE",
                timestamp=now - timedelta(hours=6),
                summary="Crude oil prices fall as supply concerns ease, benefiting transportation and consumer sectors."
            ),
            MarketNews(
                headline="Tesla Beats Delivery Estimates for Q2",
                source="Business Insider",
                category="Earnings",
                sentiment="POSITIVE",
                timestamp=now - timedelta(hours=8),
                summary="Tesla reports stronger-than-expected vehicle deliveries, boosting investor confidence."
            ),
            MarketNews(
                headline="Market Uncertainty Ahead of Earnings Season",
                source="Financial Times",
                category="Market",
                sentiment="NEUTRAL",
                timestamp=now - timedelta(hours=10),
                summary="Investors cautious as earnings season approaches, with mixed signals from economic data."
            ),
        ]

        self.market_news = news_data

    def identify_task(self, query: str) -> Dict:
        """Classify the financial query type."""
        query_lower = query.lower()

        task_mapping = {
            'price': FinanceTaskType.STOCK_PRICE,
            'stock': FinanceTaskType.STOCK_PRICE,
            'quote': FinanceTaskType.STOCK_PRICE,
            'portfolio': FinanceTaskType.PORTFOLIO_CHECK,
            'holdings': FinanceTaskType.PORTFOLIO_CHECK,
            'risk': FinanceTaskType.RISK_ASSESSMENT,
            'volatility': FinanceTaskType.RISK_ASSESSMENT,
            'trade': FinanceTaskType.TRADE_RECOMMENDATION,
            'buy': FinanceTaskType.TRADE_RECOMMENDATION,
            'sell': FinanceTaskType.TRADE_RECOMMENDATION,
            'sentiment': FinanceTaskType.MARKET_SENTIMENT,
            'news': FinanceTaskType.MARKET_SENTIMENT,
            'pe': FinanceTaskType.FINANCIAL_RATIO,
            'ratio': FinanceTaskType.FINANCIAL_RATIO,
            'dividend': FinanceTaskType.FINANCIAL_RATIO,
            'invest': FinanceTaskType.INVESTMENT_ADVICE,
            'advice': FinanceTaskType.INVESTMENT_ADVICE,
        }

        for keyword, task_type in task_mapping.items():
            if keyword in query_lower:
                return {'task_type': task_type, 'confidence': 0.9}

        return {'task_type': FinanceTaskType.GENERAL, 'confidence': 0.6}

    def _get_stock(self, symbol: str) -> Optional[Dict]:
        """Get stock data by symbol."""
        symbol_upper = symbol.upper()
        if symbol_upper in self.stocks:
            return self.stocks[symbol_upper].to_dict()
        return None

    def _get_portfolio(self, name: str) -> Optional[Dict]:
        """Get portfolio by name."""
        for portfolio_name, portfolio in self.portfolios.items():
            if name.lower() in portfolio_name.lower():
                return portfolio.to_dict()
        return None

    def _get_stocks_by_sector(self, sector: str) -> List[Dict]:
        """Get stocks by sector."""
        results = []
        for stock in self.stocks.values():
            if sector.lower() in stock.sector.lower():
                results.append(stock.to_dict())
        return results

    def _calculate_portfolio_value(self, portfolio: Portfolio) -> float:
        """Calculate total portfolio value."""
        total = portfolio.cash
        for symbol, shares in portfolio.stocks.items():
            if symbol in self.stocks:
                total += shares * self.stocks[symbol].current_price
        return total

    def _get_portfolio_performance(self, portfolio: Portfolio) -> Dict:
        """Calculate portfolio performance metrics."""
        total_value = self._calculate_portfolio_value(portfolio)
        stock_values = {}
        for symbol, shares in portfolio.stocks.items():
            if symbol in self.stocks:
                stock_values[symbol] = shares * self.stocks[symbol].current_price

        return {
            'total_value': total_value,
            'cash': portfolio.cash,
            'stock_values': stock_values,
            'stock_allocations': {s: v/total_value for s, v in stock_values.items()},
            'num_positions': len(portfolio.stocks)
        }

    def _get_market_sentiment(self) -> Dict:
        """Analyze market sentiment from news."""
        sentiments = {'POSITIVE': 0, 'NEGATIVE': 0, 'NEUTRAL': 0}
        for news in self.market_news:
            sentiments[news.sentiment] += 1

        total = len(self.market_news)
        if total == 0:
            return {'overall': 'NEUTRAL', 'positive_pct': 0, 'negative_pct': 0}

        return {
            'overall': 'POSITIVE' if sentiments['POSITIVE'] > sentiments['NEGATIVE'] else 'NEGATIVE' if sentiments['NEGATIVE'] > sentiments['POSITIVE'] else 'NEUTRAL',
            'positive_pct': sentiments['POSITIVE'] / total * 100,
            'negative_pct': sentiments['NEGATIVE'] / total * 100,
            'neutral_pct': sentiments['NEUTRAL'] / total * 100,
            'total_news': total
        }

    def process_query(self, query: str) -> FinancialResponse:
        """
        Main processing method for the Financial Analysis Agent.
        """
        print(f"\n💰 Financial Query: {query}")
        print("-" * 50)

        # Step 1: Identify task type
        task_decision = self.identify_task(query)
        task_type = task_decision['task_type']
        confidence = task_decision['confidence']

        print(f"📋 Task Identification: {task_type.value} (confidence: {confidence:.2%})")

        # Step 2: Extract symbols and names
        symbols = re.findall(r'[A-Z]{1,5}', query.upper())
        symbols = [s for s in symbols if s in self.stocks]

        portfolio_match = re.search(r'(Tech Growth Portfolio|Retirement 2050|Aggressive Trading)', query)
        portfolio_name = portfolio_match.group(0) if portfolio_match else None

        # Step 3: Retrieve data based on task type
        data = {}
        action_taken = ""
        recommendations = []
        risk_level = "MEDIUM"

        if task_type == FinanceTaskType.STOCK_PRICE:
            if symbols:
                stock_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        stock_data.append(stock)
                data['stocks'] = stock_data
                action_taken = f"Retrieved price data for {', '.join(symbols)}"
            else:
                data['all_stocks'] = [stock.to_dict() for stock in self.stocks.values()]
                action_taken = "Retrieved all stock prices"

        elif task_type == FinanceTaskType.PORTFOLIO_CHECK:
            if portfolio_name:
                portfolio = self._get_portfolio(portfolio_name)
                if portfolio:
                    data['portfolio'] = portfolio
                    # Get performance metrics
                    for p in self.portfolios.values():
                        if portfolio_name.lower() in p.name.lower():
                            perf = self._get_portfolio_performance(p)
                            data['performance'] = perf
                            break
                    action_taken = f"Retrieved portfolio: {portfolio_name}"
                else:
                    data['all_portfolios'] = [p.to_dict() for p in self.portfolios.values()]
                    data['performance'] = {
                        p.name: self._get_portfolio_performance(p)
                        for p in self.portfolios.values()
                    }
                    action_taken = "Retrieved all portfolios"
            else:
                data['all_portfolios'] = [p.to_dict() for p in self.portfolios.values()]
                data['performance'] = {
                    p.name: self._get_portfolio_performance(p)
                    for p in self.portfolios.values()
                }
                action_taken = "Retrieved all portfolios"

        elif task_type == FinanceTaskType.RISK_ASSESSMENT:
            if symbols:
                risk_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        # Calculate risk metrics
                        price_range = stock['fifty_two_week_high'] - stock['fifty_two_week_low']
                        volatility = price_range / stock['fifty_two_week_low'] * 100
                        risk_data.append({
                            'symbol': symbol,
                            'volatility': f"{volatility:.1f}%",
                            'current_position': f"{stock['current_price']}/{stock['fifty_two_week_high']} ({stock['current_price']/stock['fifty_two_week_high']*100:.1f}% of high)",
                            'risk_level': 'HIGH' if volatility > 30 else 'MEDIUM' if volatility > 20 else 'LOW'
                        })
                data['risk_analysis'] = risk_data
                action_taken = f"Assessed risk for {', '.join(symbols)}"
            elif portfolio_name:
                for p in self.portfolios.values():
                    if portfolio_name.lower() in p.name.lower():
                        perf = self._get_portfolio_performance(p)
                        # Calculate portfolio risk (simplified)
                        avg_volatility = 0
                        for symbol in p.stocks:
                            if symbol in self.stocks:
                                stock = self.stocks[symbol]
                                price_range = stock.fifty_two_week_high - stock.fifty_two_week_low
                                avg_volatility += (price_range / stock.fifty_two_week_low * 100) * (p.stocks[symbol] / sum(p.stocks.values()))
                        data['portfolio'] = p.to_dict()
                        data['portfolio_risk'] = {
                            'volatility': f"{avg_volatility:.1f}%",
                            'risk_level': 'HIGH' if avg_volatility > 30 else 'MEDIUM' if avg_volatility > 20 else 'LOW',
                            'num_positions': len(p.stocks)
                        }
                        action_taken = f"Assessed risk for portfolio: {portfolio_name}"
                        break
            else:
                data['risk_summary'] = "Please specify a stock symbol or portfolio name"
                action_taken = "Risk assessment requires symbol or portfolio"

        elif task_type == FinanceTaskType.TRADE_RECOMMENDATION:
            if symbols:
                trade_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        # Simple trade analysis
                        price_position = (stock['current_price'] - stock['fifty_two_week_low']) / (stock['fifty_two_week_high'] - stock['fifty_two_week_low']) * 100
                        if price_position < 30:
                            recommendation = "BUY"
                            reason = "Trading near 52-week low"
                        elif price_position > 70:
                            recommendation = "SELL"
                            reason = "Trading near 52-week high"
                        else:
                            recommendation = "HOLD"
                            reason = "Mid-range valuation"
                        trade_data.append({
                            'symbol': symbol,
                            'recommendation': recommendation,
                            'reason': reason,
                            'price_position': f"{price_position:.0f}% of 52-week range"
                        })
                data['trade_recommendations'] = trade_data
                action_taken = f"Generated trade recommendations for {', '.join(symbols)}"
            else:
                data['trade_advice'] = "Please specify a stock symbol for trade recommendations"
                action_taken = "Trade recommendation requires symbol"

        elif task_type == FinanceTaskType.MARKET_SENTIMENT:
            sentiment = self._get_market_sentiment()
            data['market_sentiment'] = sentiment
            data['recent_news'] = [{
                'headline': n.headline,
                'source': n.source,
                'sentiment': n.sentiment,
                'summary': n.summary
            } for n in self.market_news[:3]]
            action_taken = "Retrieved market sentiment and news"

        elif task_type == FinanceTaskType.FINANCIAL_RATIO:
            if symbols:
                ratio_data = []
                for symbol in symbols:
                    stock = self._get_stock(symbol)
                    if stock:
                        ratio_data.append({
                            'symbol': symbol,
                            'company': stock['company_name'],
                            'P/E Ratio': stock['pe_ratio'],
                            'Dividend Yield': f"{stock['dividend_yield']:.2f}%",
                            'Market Cap': f"${stock['market_cap']:.1e}",
                            'Volume': f"{stock['volume']:,}",
                            'PE vs Sector': 'Above average' if stock['pe_ratio'] > 25 else 'Below average' if stock['pe_ratio'] < 15 else 'Average'
                        })
                data['financial_ratios'] = ratio_data
                action_taken = f"Retrieved financial ratios for {', '.join(symbols)}"
            else:
                data['ratios'] = "Please specify a stock symbol for financial ratios"
                action_taken = "Financial ratio check requires symbol"

        elif task_type == FinanceTaskType.INVESTMENT_ADVICE:
            if portfolio_name:
                for p in self.portfolios.values():
                    if portfolio_name.lower() in p.name.lower():
                        perf = self._get_portfolio_performance(p)
                        data['portfolio'] = p.to_dict()
                        data['performance'] = perf
                        data['advice_context'] = f"Portfolio: {portfolio_name}, Value: ${perf['total_value']:.2f}"
                        break
            else:
                data['advice_context'] = "General investment advice requested"
                data['market_context'] = self._get_market_sentiment()
            action_taken = "Generated investment advice"

        else:  # GENERAL
            data['message'] = "General financial inquiry received"
            action_taken = "General query processed"

        # Step 4: Generate reasoning using Inkling
        reasoning_prompt = self._generate_reasoning_prompt(query, task_type, data)
        inkling_response = self.inkling.query(reasoning_prompt)

        print(f"🧠 Inkling Reasoning: {inkling_response.content[:200]}...")

        # Step 5: Build response
        return FinancialResponse(
            task_type=task_type,
            action_taken=action_taken,
            data=data,
            reasoning=inkling_response.content,
            recommendations=self._extract_recommendations(inkling_response.content),
            confidence=confidence,
            risk_level=risk_level,
            timestamp=datetime.now().isoformat()
        )

    def _generate_reasoning_prompt(self, query: str, task_type: FinanceTaskType, data: Dict) -> str:
        """Generate a prompt for Inkling to reason about the financial situation."""
        return f"""
        You are a Financial Analysis AI Assistant providing investment and trading advice.

        User Query: "{query}"
        Task Type: {task_type.value}

        Available Financial Data:
        {json.dumps(data, indent=2, default=str)}

        Please provide:
        1. A clear financial analysis of the situation
        2. Key insights from the data
        3. Recommended actions or decisions
        4. Risk assessment and considerations
        5. A concise summary for the investor/trader

        Your response should be professional, data-driven, and actionable.
        Always include a disclaimer that this is AI-assisted analysis and not personalized financial advice.
        """

    def _extract_recommendations(self, reasoning: str) -> List[str]:
        """Extract recommendations from the reasoning text."""
        recommendations = []
        lines = reasoning.split('\n')
        for line in lines:
            if any(keyword in line.lower() for keyword in ['recommend', 'advise', 'suggest', 'should', 'consider']):
                cleaned = line.strip().lstrip('•*-+0123456789. ')
                if len(cleaned) > 10 and cleaned not in recommendations:
                    recommendations.append(cleaned[:200])
        return recommendations[:5]

    def handle_query(self, query: str) -> str:
        """
        Simplified method to handle a query and return a readable response.
        """
        response = self.process_query(query)

        output = []
        output.append("=" * 60)
        output.append("💰 Financial Analysis Response")
        output.append("=" * 60)
        output.append(f"📋 Task: {response.task_type.value}")
        output.append(f"✅ Action: {response.action_taken}")
        output.append(f"🔒 Confidence: {response.confidence:.2%}")
        output.append(f"⚠️ Risk Level: {response.risk_level}")
        output.append("-" * 60)
        output.append("📊 Analysis:")
        output.append(response.reasoning)
        if response.recommendations:
            output.append("-" * 60)
            output.append("💡 Recommendations:")
            for i, rec in enumerate(response.recommendations, 1):
                output.append(f"   {i}. {rec}")
        output.append("=" * 60)
        output.append("⚠️ DISCLAIMER: This is AI-assisted financial analysis.")
        output.append("   Always consult with a qualified financial advisor.")
        output.append("   This system does not replace professional financial advice.")
        output.append("=" * 60)

        return "\n".join(output)


# ============================================================
# PART 4: DEMONSTRATION
# ============================================================

def run_finance_demo():
    """Demonstrate the Financial Analysis Agent in action."""
    print("\n" + "="*60)
    print("💰  FINANCIAL ANALYSIS AGENT")
    print("   Complete Agentic Solution for Financial Analysis & Trading")
    print("="*60 + "\n")

    agent = FinancialAnalysisAgent()

    test_queries = [
        "What is the current price of AAPL and TSLA?",
        "Show me the Tech Growth Portfolio",
        "Assess risk for MSFT and AMZN",
        "Should I buy or sell GOOGL?",
        "What is the market sentiment today?",
        "What are the financial ratios for JPM?",
    ]

    for i, query in enumerate(test_queries[:5], 1):
        print(f"\n{'='*60}")
        print(f"TEST {i}")
        print('='*60)
        result = agent.handle_query(query)
        print(result)


# ============================================================
# PART 5: INTERACTIVE FINANCIAL CHAT
# ============================================================

def interactive_finance():
    """Interactive chat with the Financial Analysis Agent."""
    print("\n" + "="*60)
    print("💰  FINANCIAL AGENT - Interactive Investment Analysis")
    print("="*60)
    print("Type your query below. Type 'exit' to quit.")
    print("")
    print("📚 Sample Queries:")
    print("   • What is the price of AAPL?")
    print("   • Show me the Tech Growth Portfolio")
    print("   • Assess risk for MSFT")
    print("   • Should I buy or sell GOOGL?")
    print("   • What is the market sentiment today?")
    print("   • Financial ratios for JPM")
    print("   • Investment advice for Retirement 2050")
    print("-"*60 + "\n")
    print("⚠️ DISCLAIMER: This is AI-assisted financial analysis only.")
    print("   Always consult with a qualified financial advisor.")
    print("   This system does not replace professional financial advice.")
    print("-"*60 + "\n")

    agent = FinancialAnalysisAgent()

    while True:
        try:
            query = input("\n💰 You: ").strip()
            if query.lower() in ['exit', 'quit', 'q']:
                print("\n👋 Financial Agent signing off!")
                break

            if not query:
                continue

            result = agent.handle_query(query)
            print(result)

        except KeyboardInterrupt:
            print("\n\n👋 Financial Agent signing off!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")


# ============================================================
# PART 6: CONFIGURATION
# ============================================================

config = {
    'gemma_model': 'frankmorales2020/gemma-4-e4b-stl10-topo-2026',
    'gemma_base': 'frankmorales2020/gemma-4-e4b-unesco-optimized',
    'inkling_model': INKLING_MODEL_ID,
    'inkling_max_tokens': INKLING_MAX_TOKENS,
    'inkling_temperature': INKLING_TEMPERATURE,
    'api_key_configured': bool(OPENROUTER_API_KEY),
    'system': 'Financial Analysis Agent',
    'version': '1.0.0',
    'stocks': 8,
    'portfolios': 3,
    'news_items': 5
}

print("\n📦 Configuration:")
print(json.dumps(config, indent=2))


✅ API key loaded from Colab secrets! Ending: ****f808

📦 Configuration:
{
  "gemma_model": "frankmorales2020/gemma-4-e4b-stl10-topo-2026",
  "gemma_base": "frankmorales2020/gemma-4-e4b-unesco-optimized",
  "inkling_model": "thinkingmachines/inkling",
  "inkling_max_tokens": 2048,
  "inkling_temperature": 0.7,
  "api_key_configured": true,
  "system": "Financial Analysis Agent",
  "version": "1.0.0",
  "stocks": 8,
  "portfolios": 3,
  "news_items": 5
}


In [3]:
# ============================================================
# FINANCIAL ANALYSIS AGENT - CORRECT USAGE (NO DUPLICATION)
# ============================================================

# Initialize the agent
agent = FinancialAnalysisAgent()

# Financial queries - these are the actual commands to execute
agent.handle_query("What is the current price of AAPL and TSLA?")
agent.handle_query("Show me the Tech Growth Portfolio")
agent.handle_query("Assess risk for MSFT and AMZN")
agent.handle_query("Should I buy or sell GOOGL?")
agent.handle_query("What is the market sentiment today?")

# ✅ That's it! No duplicate interactive_finance() needed.
# The 5 queries above are what you want to run.


#OR
#interactive_finance()


💰  FINANCIAL ANALYSIS AGENT

📥 Loading Gemma-4 TOPO-2026 Certified Model...
   Model: frankmorales2020/gemma-4-e4b-stl10-topo-2026
   Device: cuda

📥 Loading tokenizer...
   ✅ Tokenizer loaded. Vocab size: 262144

👁️ Loading vision model...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

   ✅ Gemma loaded (Unsloth)

📥 Downloading trained weights...
   ✅ Checkpoint loaded! Task C: 100.0%

🏗️ Building classifier...
   ✅ Model ready!

📊 Certification:
   standard: TOPO-2026
   runs: 5/5
   task_c_accuracy: 100.0%
   forgetting: 0.48%
   s_narrow: 5.970999999965
   status: ✅ CERTIFIED
✅ Inkling client initialized
   Model: thinkingmachines/inkling
   Max Tokens: 2048
   Temperature: 0.7
✅ Financial Analysis Agent Ready!
   📈 Stocks: 8
   💼 Portfolios: 3
   📰 News: 5


💰 Financial Query: What is the current price of AAPL and TSLA?
--------------------------------------------------
📋 Task Identification: stock_price (confidence: 90.00%)
🧠 Inkling Reasoning: **AI-Assisted Financial Analysis — AAPL & TSLA**
*Data as of: 2026-08-04 12:31 | Source: Provided Market Data*

---

### 1. Financial Analysis of the Situation

**Apple Inc. (AAPL) — Technology | Larg...

💰 Financial Query: Show me the Tech Growth Portfolio
--------------------------------------------------
📋 Task Identif

'============================================================\n💰 Financial Analysis Response\n============================================================\n📋 Task: market_sentiment\n✅ Action: Retrieved market sentiment and news\n🔒 Confidence: 90.00%\n⚠️ Risk Level: MEDIUM\n------------------------------------------------------------\n📊 Analysis:\n**1. Financial Analysis of Current Market Sentiment**\n\nThe data indicates a **broadly positive market environment**, with an overall sentiment score of **POSITIVE** supported by a **60% positive / 20% negative / 20% neutral** distribution across 5 news items. This represents a 3:1 bullish-to-bearish news ratio, suggesting risk-on conditions are prevailing.\n\nKey macro and sector dynamics observed:\n*   **Monetary Policy Tailwind:** The Federal Reserve is signaling potential rate cuts as inflation cools. This is a structurally supportive development for equity valuations, particularly for growth-oriented and rate-sensitive assets (e.g., tech